In [1]:
from optim_utils import * 
import os
import logging
import torch
import argparse
from tqdm.auto import tqdm
from prompt2prompt import *
from ptp_utils import load_512, load_stable_diffusion, run_and_display
from torchvision.utils import save_image, make_grid
from run_image_editing import NullInversion, get_logger
from clip_interrogator import Config, Interrogator

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

args = argparse.Namespace()
args.prompt_len = 4
args.iters = 3000
args.lr = 0.1
args.weight_decay = 0.1
args.prompt_bs = 1
args.loss_weight = 1.0
args.print_step = 100
args.batch_size = 1
args.clip_model = 'ViT-H-14'
args.clip_pretrain = 'laion2b_s32b_b79k'



In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# customized parameters
args.orig_path = "./data/n000164/set_A/"
args.save_path = "./data/n000164/set_A_edited/"
args.log_name = "test"
args.resume_name = None


args.device = device
args.my_token = ''
args.low_resource = False
args.num_ddim_steps = 50
args.guidance_scale = 7.5
args.max_num_words = 77

In [3]:
os.makedirs(args.save_path, exist_ok=True)
image_paths = [os.path.join(args.orig_path, sub_dir) for sub_dir in os.listdir(args.orig_path)]
save_paths = [os.path.join(args.save_path, sub_dir) for sub_dir in os.listdir(args.orig_path)]

orig_images = [Image.open(image_path).convert("RGB") for image_path in image_paths]
offsets = (0,0,0,0)
connect = " , "

prefix, prompts_clip, learned_prompt = [], [], None

ci = Interrogator(Config(clip_model_name="ViT-L-14/openai"))
for orig_image in orig_images:
    prompt_clip = ci.interrogate(orig_image)
    print(prompt_clip)
    prompts_clip.append(prompt_clip)

    prefix_clip = prompt_clip.split(',')[0]
    print(prefix_clip)
    prefix.append(prefix_clip)

model, _, preprocess = open_clip.create_model_and_transforms(args.clip_model, pretrained=args.clip_pretrain, device=device)
learned_prompt = optimize_prompt(model, preprocess, args, device, 
                target_images=orig_images, prefix=[i + connect for i in prefix])

Loading BLIP model...
load checkpoint from https://storage.googleapis.com/sfr-vision-language-research/BLIP/models/model_large_caption.pth
Loading CLIP model...
Loaded CLIP model and data in 5.25 seconds.


100%|██████████| 50/50 [00:00<00:00, 288.35it/s]


a close up of a person wearing a hat, brown stubble, formula one car, walking confidently, smiling, he has yellow wolf eyes, action sports, tyler west, biggish nose, open wings, hypermasculine, website, he has dark grey hairs, idol, a man, 7
a close up of a person wearing a hat


100%|██████████| 50/50 [00:00<00:00, 168.47it/s]


a close up of a person wearing a hat and sunglasses, skiing, has black wings, positive mood, ice-blue-eyes, energy drink, maxxis, confident expression, fantasy list, interview, small rectangular glasses, the gunslinger, buggy
a close up of a person wearing a hat and sunglasses


100%|██████████| 50/50 [00:00<00:00, 283.33it/s]


a close up of a person wearing a hat, blond brown stubble thin beard, charming sly smile, kerlee, italian, dominik mayer, red bull, associated press photography, myserious man, sarcastic smiling, lance, west slav features, roman gladiator, sarcastic smile, dull, loosely cropped, myazaki
a close up of a person wearing a hat


100%|██████████| 50/50 [00:00<00:00, 313.02it/s]


a close up of a person wearing a hat, christian horner portrait, very attractive man with beard, with a seductive smile, crow, associated press photo, inspired by Kees Maks, expansive, vale, imet2020, alpine, eyecandy, 3 2 - year - old, swashbuckler, hasselblatt
a close up of a person wearing a hat

step: 0, lr: 0.1, cosim: 0.150, text: wireless andme navigating hindi 

step: 100, lr: 0.1, cosim: 0.271, text: ofthesachinrunnerpolo 

step: 200, lr: 0.1, cosim: 0.310, text: councill✔️indycar footballer 

step: 300, lr: 0.1, cosim: 0.284, text: yuvricciardo olympics instructors 

step: 400, lr: 0.1, cosim: 0.248, text: imrankhanpti ✌️ olympics renowned 

step: 500, lr: 0.1, cosim: 0.289, text: coldplay pistorfootballers advisor 

step: 600, lr: 0.1, cosim: 0.240, text: shift�cricket founder 

step: 700, lr: 0.1, cosim: 0.314, text: avicii pistormotorsports founder 

step: 800, lr: 0.1, cosim: 0.231, text: seac❤️❤️ skiing and 

step: 900, lr: 0.1, cosim: 0.279, text: coldplay 😎😎 motorsport

In [4]:
#####################
#   image editing   #
#####################

# null inversion
# load stable diffusion model
pipe, tokenizer = load_stable_diffusion(args)

x_t_list, uncond_embeddings_list = [], []
for i in range(len(prefix)):
    prompt = prefix[i] + connect + learned_prompt
    # start inversion
    null_inversion = NullInversion(args, pipe)
    (image_gt, image_enc), x_t, uncond_embeddings = null_inversion.invert(image_paths[i], prompt, offsets=offsets, verbose=True)
    x_t_list.append(x_t)
    uncond_embeddings_list.append(uncond_embeddings)

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

DDIM inversion...
Null-text optimization...


  0%|          | 0/500 [00:00<?, ?it/s]

DDIM inversion...
Null-text optimization...


  0%|          | 0/500 [00:00<?, ?it/s]

DDIM inversion...
Null-text optimization...


  0%|          | 0/500 [00:00<?, ?it/s]

DDIM inversion...
Null-text optimization...


  0%|          | 0/500 [00:00<?, ?it/s]

In [6]:
for i in range(len(prefix)):
    prompt = prefix[i] + connect + learned_prompt
    x_t, uncond_embeddings = x_t_list[i], uncond_embeddings_list[i]
    prompt_target = prompt + connect + \
    "with a mustache , wearing headphones"
    # "serious face" "angry"
    
    # "wearing transparent glasses , wearing round frame glasses"
    
    # "wearing an eye mask"
    # "with a mustache"
    # "wearing headphones"
    # "wearing sunglasses", n000089, n000164, n000243
    # "eyes closed", n000181, n000217
    
    # "dyed hair"

    # "bald"
    # "curly hair"
    # "thick beard"
    # "big beard"
    # "holding a cigarette"
    # "with pirate eyepatch"
    # "masquerade"

    # image editing
    prompts = [
        prompt,         # original prompt
        prompt_target   # target prompt
    ]
    cross_replace_steps = {'default_': .8, }
    self_replace_steps = .2
    blend = 'man' if 'man' in prompt else 'person'
    blend_word = None
    eq_params = None
    controller = make_controller(args, pipe.tokenizer, prompts, False, cross_replace_steps, self_replace_steps, blend_word, eq_params)

    images, _ = run_and_display(args, pipe, prompts, controller, run_baseline=False, latent=x_t, uncond_embeddings=uncond_embeddings)
    if args.log_name == 'test':
        # save comparison
        tensors = [torch.from_numpy(img).permute(2, 0, 1).float() / 255. for img in \
                [images[0], images[1]]]
        grid = make_grid(tensors, nrow=2)
        save_image(grid, f'{args.save_path}/combined_image_{i}.png')
        

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]